# 3. Where Python says no

Sections 5, 5b and 6 of `demo/dci_demo.py`, followed by the overlap of two live contexts reproduced without threads. The failures are asserted in `tests/unit/test_limits.py`, `test_serialized_roles.py` and `test_async_overlap.py`, and [APPROACHES.md](https://github.com/CesarBallardini/dci-in-python/blob/main/APPROACHES.md#context--the-role-mixed-into-the-objects-class) explains each one.

The first cell installs `banking` when the notebook runs in the browser. On the desktop it is already installed in the project environment, and the cell only reports the runtime.

In [ ]:
import sys

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(['banking'])

print('Python', sys.version.split()[0], 'on', sys.platform)

In [ ]:
import asyncio
import threading
from dataclasses import dataclass

from banking.contexts.money_transfer import (
    InstanceMoneyTransferContext,
    MoneyTransferContext,
    SerializedMoneyTransferContext,
    TransparentMoneyTransferContext,
)
from banking.data.account import Account
from banking.data.money import Money

BINDERS = {
    'Context': MoneyTransferContext,
    'TransparentContext': TransparentMoneyTransferContext,
    'SerializedContext': SerializedMoneyTransferContext,
    'InstanceContext': InstanceMoneyTransferContext,
}


def d(title):
    print(f'\n{"=" * 62}\n{title}\n{"=" * 62}')

## `__slots__`

CPython reassigns `__class__` only between classes with the same object layout, and the synthesized class does not share the layout of a slotted one.

In [ ]:
d('5. Where class binding fails: __slots__')


class SlottedAccount:
    __slots__ = ('balance', 'holder')

    def __init__(self, holder, balance):
        self.holder, self.balance = holder, balance

    def debit(self, m):
        self.balance = self.balance - m

    def credit(self, m):
        self.balance = self.balance + m


try:
    MoneyTransferContext(SlottedAccount('Alice', Money(100)), SlottedAccount('Bob', Money(0)), Money(10)).execute()
except TypeError as e:
    print('TypeError:', e)

## A frozen dataclass

The same outcome by a different route: a frozen dataclass rejects every attribute assignment, and `__class__` is an attribute.

In [ ]:
d('5b. The same failure, reached by the idiomatic value object')


@dataclass(frozen=True)
class FrozenAccount:
    holder: str
    balance: Money

    def debit(self, m):
        object.__setattr__(self, 'balance', self.balance - m)

    def credit(self, m):
        object.__setattr__(self, 'balance', self.balance + m)


try:
    MoneyTransferContext(FrozenAccount('Alice', Money(100)), FrozenAccount('Bob', Money(0)), Money(10)).execute()
except Exception as e:
    print(f'{type(e).__name__}: {e}')
print('the field it refuses to assign is __class__')

## Two live contexts over one object

The class of an object is state of the object, and a context uses it as if it were state of the context. A second context asking for the same role cannot build a consistent MRO.

In [ ]:
d('6. Where class binding fails: two live contexts over one object')
shared = Account('Erin', Money(1_000_000))
other = Account('Frank', Money(0))
third = Account('Grace', Money(0))

c1 = MoneyTransferContext(shared, other, Money(10))
c2 = MoneyTransferContext(shared, third, Money(20))
c1.__enter__()
print('with c1 open: type(shared).__name__ =', type(shared).__name__)
try:
    c2.__enter__()
except TypeError as e:
    print('c2.__enter__() ->', type(e).__name__ + ':', e)
c1.__exit__()
print('after closing c1:', type(shared).__name__)

## The same overlap, served concurrently

`demo/dci_demo.py` serves this case with two threads. The browser runtime has none: Pyodide builds CPython without pthreads, so starting a thread raises `RuntimeError`. On the desktop the next cell starts one.

In [ ]:
try:
    worker = threading.Thread(target=lambda: None)
    worker.start()
    worker.join()
    print('this runtime starts threads')
except RuntimeError as exc:
    print(f'RuntimeError: {exc}')

The overlap does not depend on threads. It needs one object inside two live contexts at once, and two `asyncio` tasks suspended at an `await` inside their contexts reach that on a single thread, in the browser and on the desktop alike. Each task has its own copy of the `ContextVar` state, as each thread does. The class of the shared account is shared between the tasks, as it is between threads.

The tasks order themselves with `asyncio.Event`, so the interleaving is fixed by the code rather than by timing. These are the scenarios `tests/unit/test_async_overlap.py` asserts.

In [ ]:
async def same_role(binder):
    shared = Account('Erin', Money(1_000_000))
    inside = asyncio.Event()
    outcome = 'both transfers complete'

    async def slow():
        held = binder(shared, Account('Frank', Money(0)), Money(5))
        with held:
            inside.set()
            await asyncio.sleep(0)  # the other task opens its context in this gap
            held.run_script()

    async def fast():
        nonlocal outcome
        await inside.wait()
        try:
            binder(shared, Account('Grace', Money(0)), Money(7)).execute()
        except (TypeError, RuntimeError) as exc:
            outcome = f'the second task is refused: {type(exc).__name__}'

    await asyncio.gather(slow(), fast())
    return outcome


async def different_roles(binder):
    shared = Account('Erin', Money(1_000_000))
    paying_out_inside, paid_into_ran = asyncio.Event(), asyncio.Event()

    async def paying_out():
        held = binder(shared, Account('Frank', Money(0)), Money(5))
        with held:
            paying_out_inside.set()
            await paid_into_ran.wait()
            held.run_script()

    async def paid_into():
        await paying_out_inside.wait()
        held = binder(Account('Grace', Money(1000)), shared, Money(7))
        with held:
            held.run_script()
            paid_into_ran.set()
            await asyncio.sleep(0)  # the other transfer runs and leaves inside this gap

    await asyncio.gather(paying_out(), paid_into())
    left = 'still has .transfer' if hasattr(shared, 'transfer') else 'is itself again'
    return f'balance {shared.balance.amount}, and after both left the account {left}'

In [ ]:
print('Two tasks, the same role:')
for name, binder in BINDERS.items():
    print(f'  {name:<20} {await same_role(binder)}')
print('Two tasks, different roles, overlapping without nesting:')
for name, binder in BINDERS.items():
    print(f'  {name:<20} {await different_roles(binder)}')

Every binder behaves as it does under threads except `SerializedContext`. Its lock is an `RLock`, which belongs to a thread, and every task runs on the same thread, so the second task re-enters the lock instead of waiting for it. Under threads the same two scenarios complete both transfers and leave the account intact. `SerializedContext` serializes threads, not tasks: none of the four binders holds a lock an event loop can wait on.